In [1]:
import pandas as pd
import numpy as np

# İçinde eksik ve aykırı değerler olan örnek bir veri seti oluşturalım
data = {
    'Yas': [25, 30, np.nan, 22, 150, 28, 35, np.nan, 40, 29], # 150 bir aykırı değer, 2 tane eksik (NaN) var
    'Maas': [100, 6000, 5500, np.nan, 7000, 150000, 6500, 6200, 7100, 5800], # 150.000 aykırı değer, 1 eksik var
    'Departman': ['IT', 'IK', 'IT', 'Satis', 'IT', 'Satis', np.nan, 'IK', 'IT', 'Satis'] # 1 eksik var
}

In [ ]:
data

In [ ]:
df = pd.DataFrame(data)

In [ ]:
print("--- ORİJİNAL VERİ SETİ ---")
print(df, "\n")

In [ ]:
# --- EKSİK VERİLERİ DOLDURMA YÖNTEMLERİ ---

df_temiz = df.copy()

In [ ]:
# 1. Sayısal Veriyi Medyan ile Doldurmak (Yas sütunundaki outlier'dan etkilenmemek için medyan seçtik)
yas_medyan = df_temiz['Yas'].median()
df_temiz['Yas']
yas_medyan

In [ ]:
df_temiz['Yas'] = df_temiz['Yas'].fillna(yas_medyan)

In [ ]:
df_temiz['Yas'][0]

In [ ]:
print(df_temiz, "\n")

In [ ]:
df_temiz['Departman'][1] = 'Satis'


In [ ]:
df_temiz

In [ ]:
df_temiz['Departman'].mode()

In [ ]:
departman_mod = df_temiz['Departman'].mode()[1]
print(departman_mod)

In [ ]:
# 2. Kategorik Veriyi Mod (En çok tekrar eden) ile Doldurmak
departman_mod = df_temiz['Departman'].mode()[0]
df_temiz['Departman'] = df_temiz['Departman'].fillna(departman_mod)

In [ ]:
print(df_temiz, "\n")

In [ ]:
# 3. Satır Silmek (Sadece 'Maas' bilgisi eksik olan satırı veriden atalım)
df_temiz = df_temiz.dropna(subset=['Maas'])

In [ ]:
print("--- EKSİK VERİLER ÇÖZÜLDÜKTEN SONRA ---")
print(df_temiz, "\n")

In [ ]:
df_temiz.shape

# 2. Aykırı Değer Yönetimi (Outlier Handling)
Aykırı değerleri tespit etmenin en yaygın yolu IQR (Interquartile Range - Çeyrekler Açıklığı) yöntemidir. Bu yöntem, veriyi %25'lik dilimlere böler. Alt çeyrek (Q1) ve Üst çeyrek (Q3) arasındaki fark (IQR) hesaplanır. Bu aralığın 1.5 katından daha aşağıda veya daha yukarıda olan değerler "aykırı" kabul edilir.
<img src="iqr.png" width="1000" height="400">


# --- AYKIRI DEĞERLERİ TESPİT ETME (IQR YÖNTEMİ) ---

In [ ]:
# Maaş sütunu için Q1 (%25) ve Q3 (%75) değerlerini bulalım
Q1 = df_temiz['Maas'].quantile(0.25)
Q3 = df_temiz['Maas'].quantile(0.75)
IQR = Q3 - Q1
print('Q1',Q1)
print('Q3',Q3)

print(IQR)

In [ ]:
# Alt ve Üst Sınırları Belirleyelim
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

In [ ]:
print(f"Maaş için Alt Sınır: {alt_sinir}, Üst Sınır: {ust_sinir}\n")

In [ ]:
# --- YAKLAŞIM 1: BASKILAMA (Capping) YÖNTEMİ ---
df_temiz['Maas'][8] = 15
df_baskilanmis = df_temiz.copy()
df_baskilanmis

In [ ]:
df_baskilanmis['Maas'] < alt_sinir

In [ ]:
df_baskilanmis.loc[[0,8]] = alt_sinir

In [ ]:
df_baskilanmis

In [ ]:
# Üst sınırdan büyük olanları üst sınıra, alt sınırdan küçük olanları alt sınıra eşitle
df_baskilanmis.loc[df_baskilanmis['Maas'] > ust_sinir, 'Maas'] = ust_sinir
df_baskilanmis.loc[df_baskilanmis['Maas'] < alt_sinir, 'Maas'] = alt_sinir

In [ ]:
print("--- MAAŞ OUTLIER'I BASKILANDIKTAN SONRA (CAPPING) ---")
print(df_baskilanmis, "\n")

In [ ]:
df_temiz

In [ ]:
# --- YAKLAŞIM 2: SİLME (Trimming) YÖNTEMİ ---
df_kirpilmis = df_temiz.copy()

In [ ]:
df_kirpilmis

In [ ]:
df_kirpilmis['Maas'] >= alt_sinir

In [ ]:
df_kirpilmis['Maas'] <= ust_sinir

In [ ]:
(df_kirpilmis['Maas'] >= alt_sinir) & (df_kirpilmis['Maas'] <= ust_sinir)

In [ ]:
# Sadece alt sınır ile üst sınır arasında kalan (normal) verileri tut
df_kirpilmis = df_kirpilmis[]

In [ ]:
print("--- MAAŞ OUTLIER'I SİLİNDİKTEN SONRA (TRIMMING) ---")
print(df_kirpilmis)

In [ ]:
df_kirpilmis.shape

# kategorik verilerin sayısallaştırılması

Makine öğrenmesi modelleri temelde devasa matematiksel denklemlerdir; dolayısıyla "IT", "Satış" veya "İstanbul" gibi metinleri anlayamazlar, sadece sayıları anlarlar.

-   Kategorik verileri sayısallaştırırken verinin türüne göre iki temel yöntem kullanırız:

1.  Ordinal (Sıralı) Veriler: Aralarında bir hiyerarşi veya büyüklük-küçüklük ilişkisi olan verilerdir. (Örn: Eğitim Seviyesi -> Lise < Lisans < Yüksek Lisans). Bunları 1, 2, 3 gibi ardışık sayılara çeviririz (Label Encoding).

2.  Nominal (Sırasız) Veriler: Aralarında bir üstünlük olmayan verilerdir. (Örn: Departman -> IT, İK, Satış). Eğer bunlara 1, 2, 3 verirsek, model "Satış (3), IT'den (1) büyüktür" gibi yanlış bir matematiksel algıya kapılır. Bunu önlemek için her kategoriye özel yeni bir sütun açarız ve sadece o kategoriye aitse 1, değilse 0 yazarız. Buna One-Hot Encoding denir.

In [8]:
# ---------------------------------------------------------
# ADIM 1: ÖRNEK VERİ SETİ OLUŞTURMA
# ---------------------------------------------------------
veri = {
    'Isim': ['Ali', 'Ayse', 'Mehmet', 'Fatma', 'Can'],
    'Departman': ['IT', 'IK', 'Satis', 'IT', 'IK'], # Nominal (Sırasız)
    'Egitim': ['Lise', 'Lisans', 'Yuksek Lisans', 'Lisans', 'Lise'], # Ordinal (Sıralı)
    'Maas': [5000, 7000, 9000, 7500, 5200]
}
df = pd.DataFrame(veri)

In [9]:
print("--- 1. ORİJİNAL VERİ SETİ ---")
print(df, "\n")

--- 1. ORİJİNAL VERİ SETİ ---
     Isim Departman         Egitim  Maas
0     Ali        IT           Lise  5000
1    Ayse        IK         Lisans  7000
2  Mehmet     Satis  Yuksek Lisans  9000
3   Fatma        IT         Lisans  7500
4     Can        IK           Lise  5200 



In [ ]:
# ---------------------------------------------------------
# ADIM 2: ORDINAL (SIRALI) VERİLERİ DÖNÜŞTÜRME (Mapping)
# ---------------------------------------------------------
# Eğitim seviyelerinin hiyerarşisini biz bildiğimiz için bir sözlük (dictionary) ile eşleştiriyoruz.
egitim_sozlugu = {'Lise': 1, 'Lisans': 2, 'Yuksek Lisans': 3}
it_szlugu = {'IT': 100, 'Satis': 200, 'IK': 300}
df['Dept_categorical'] = df['Departman'].map(it_szlugu)

# Artık metin olan eski 'Egitim' sütununu silebiliriz
df = df.drop('Egitim', axis=1) 

print("--- 2. EĞİTİM SEVİYESİ SAYISALLAŞTIRILDI (Label Encoding) ---")
print(df, "\n")

--- 2. EĞİTİM SEVİYESİ SAYISALLAŞTIRILDI (Label Encoding) ---
     Isim Departman  Maas  Egitim_Sayisal
0     Ali        IT  5000               1
1    Ayse        IK  7000               2
2  Mehmet     Satis  9000               3
3   Fatma        IT  7500               2
4     Can        IK  5200               1 



In [ ]:
# ---------------------------------------------------------
# ADIM 3: NOMINAL (SIRASIZ) VERİLERİ DÖNÜŞTÜRME (One-Hot Encoding)
# ---------------------------------------------------------
# Sadece analiz amaçlı olan ve modele bir şey katmayacak 'Isim' sütununu çıkaralım
df = df.drop('Isim', axis=1)

# pd.get_dummies() ile 'Departman' sütununu 0 ve 1'lere çeviriyoruz
df_makineye_hazir = pd.get_dummies(df, columns=['Departman'], drop_first=True)

# Çıktıların True/False yerine 1/0 olması için integer'a çeviriyoruz (Pandas'ın yeni sürümleri için)
df_makineye_hazir = df_makineye_hazir.astype(int)

print("--- 3. MAKİNE ÖĞRENMESİNE TAMAMEN HAZIR VERİ SETİ ---")
print(df_makineye_hazir)